In [1]:
!pip install kagglehub tensorflow opencv-python-headless pillow -q

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os, glob, shutil, random
import kagglehub
import tensorflow as tf
from tensorflow.keras import layers

DRIVE_MODEL_DIR = "/content/drive/MyDrive/second_face_models"

os.makedirs(f"{DRIVE_MODEL_DIR}/all_epochs", exist_ok=True)
os.makedirs(f"{DRIVE_MODEL_DIR}/best", exist_ok=True)
os.makedirs(f"{DRIVE_MODEL_DIR}/final", exist_ok=True)

print("Models will be saved to:", DRIVE_MODEL_DIR)

Models will be saved to: /content/drive/MyDrive/second_face_models


In [4]:
path = kagglehub.dataset_download("manjilkarki/deepfake-and-real-images")
print("Dataset path:", path)

100%|██████████| 1.68G/1.68G [00:45<00:00, 39.6MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/manjilkarki/deepfake-and-real-images/versions/1


In [5]:
os.makedirs("dataset/authentic", exist_ok=True)
os.makedirs("dataset/deepfake", exist_ok=True)

real_paths = glob.glob(path + "/**/*Real*/*", recursive=True)
fake_paths = glob.glob(path + "/**/*Fake*/*", recursive=True)

real_paths = [p for p in real_paths if p.lower().endswith((".jpg", ".jpeg", ".png"))]
fake_paths = [p for p in fake_paths if p.lower().endswith((".jpg", ".jpeg", ".png"))]

real_paths = random.sample(real_paths, min(5000, len(real_paths)))
fake_paths = random.sample(fake_paths, min(5000, len(fake_paths)))

for i, p in enumerate(real_paths):
    shutil.copy(p, f"dataset/authentic/real_{i}.jpg")

for i, p in enumerate(fake_paths):
    shutil.copy(p, f"dataset/deepfake/fake_{i}.jpg")

print("authentic:", len(real_paths))
print("deepfake:", len(fake_paths))

authentic: 5000
deepfake: 5000


In [6]:
IMG_SIZE = 224
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

print("Class names:", train_ds.class_names)

Found 10000 files belonging to 2 classes.
Using 8000 files for training.
Found 10000 files belonging to 2 classes.
Using 2000 files for validation.
Class names: ['authentic', 'deepfake']


In [7]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(512).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [8]:
base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base.trainable = False

model = tf.keras.Sequential([
    layers.Rescaling(1./255),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.35),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [10]:
save_every_epoch = tf.keras.callbacks.ModelCheckpoint(
    filepath=DRIVE_MODEL_DIR + "/all_epochs/model_epoch_{epoch:02d}_valauc_{val_auc:.4f}.keras",
    monitor="val_auc",
    mode="max",
    save_best_only=False,
    save_weights_only=False,
    verbose=1
)

save_only_better_models = tf.keras.callbacks.ModelCheckpoint(
    filepath=DRIVE_MODEL_DIR + "/best/best_model.keras",
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=4,
    restore_best_weights=True,
    verbose=1
)

In [11]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[
        save_every_epoch,
        save_only_better_models,
        early_stop
    ]
)

Epoch 1/15
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.5893 - auc: 0.6203 - loss: 0.7044
Epoch 1: saving model to /content/drive/MyDrive/second_face_models/all_epochs/model_epoch_01_valauc_0.7601.keras

Epoch 1: finished saving model to /content/drive/MyDrive/second_face_models/all_epochs/model_epoch_01_valauc_0.7601.keras

Epoch 1: val_auc improved from None to 0.76015, saving model to /content/drive/MyDrive/second_face_models/best/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/second_face_models/best/best_model.keras
500/500 ━━━━━━━━━━━━━━━━━━━━ 49s 44ms/step - accuracy: 0.6064 - auc: 0.6451 - loss: 0.6825 - val_accuracy: 0.7010 - val_auc: 0.7601 - val_loss: 0.5879
Epoch 2/15
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6556 - auc: 0.7138 - loss: 0.6262
Epoch 2: saving model to /content/drive/MyDrive/second_face_models/all_epochs/model_epoch_02_valauc_0.7917.keras

Epoch 2: finished saving model to /content/drive/MyDrive/second_face_

In [12]:
model.save(DRIVE_MODEL_DIR + "/final/final_restored_model.keras")
print("Final model saved to:", DRIVE_MODEL_DIR + "/final/final_restored_model.keras")
print("Best model saved to:", DRIVE_MODEL_DIR + "/best/best_model.keras")

Final model saved to: /content/drive/MyDrive/second_face_models/final/final_restored_model.keras
Best model saved to: /content/drive/MyDrive/second_face_models/best/best_model.keras
